# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_comms_" + datetime_str + ".zarr"

output_file = "./results/patrolling_comms_draft2_test15_pareto_3.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 300
GPUS = [0, 1, 2, 3, 4, 5, 6, 7]

def get_gpu_least_used():
    gpu_memory = os.popen("nvidia-smi --query-gpu=memory.used --format=csv").read()
    gpu_memory = gpu_memory.split("\n")[1:-1]
    gpu_memory = [int(x.split()[0]) for x in gpu_memory]
    least_used_gpu = gpu_memory.index(min(gpu_memory))
    return least_used_gpu

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(1)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(10)
default_args["episode_length"] = wrap_arg_val(150)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)

default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

# default_args["graph_name"] = wrap_arg_val("cumberland")
# default_args["graph_file"] = wrap_arg_val("../../patrolling_zoo/graphs/cumberland.graph")
default_args["graph_random"] = wrap_arg_val(True)

default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Evaluation

In [3]:
import numpy as np
# obs_radii = [0.0, 50.0, 100.0, 150.0, 200.0, 250.0, 300.0]
obs_radii = np.arange(5.0, 51.0, 5.0).tolist()
comms_probabilities = np.arange(0.0, 0.51, 0.05).tolist()

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    process_args = []

    def add_process(args):
        process_args.append(args)
    
    def start_process(args):
        p = Process(target=main, args=(args,))
        p.start()
        return p

    for obs_radius in obs_radii:
        for comms_p in comms_probabilities:
            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-draft1/wandb/run-20260513_145517-q92oa1om/files", default_args)
            args["eval_output_file"] = wrap_arg_val(output_file)
            args["eval_series"] = "MAGEC"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            add_process(args)

            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-draft1/wandb/run-20260513_212509-zyxc9sde/files", default_args)
            args["eval_output_file"] = wrap_arg_val(output_file)
            args["eval_series"] = "Prediction"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            add_process(args)

            # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-uqstd-draft1/wandb/run-20260513_145517-mmr9jq7r/files", default_args)
            # args["eval_series"] = "Prediction + UQ"
            # args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            # args["observation_radius"] = wrap_arg_val(obs_radius)
            # args["communication_probability"] = wrap_arg_val(comms_p)
            # add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260513_145518-yelbuvdd/files", default_args)
        args["eval_series"] = f"Prediction+UQ and Learned Observation-Based Comms."
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["ex_env_communication_mode"] = wrap_arg_val("none")
        add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260513_145518-yelbuvdd/files", default_args)
        args["eval_series"] = f"Prediction+UQ and Learned Prediction+UQ-Based Comms."
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        add_process(args)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(process_args) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(process_args) > 0:
            args = process_args.pop(0)

            # Get least used GPU and set it in args.
            least_used_gpu = get_gpu_least_used()
            args["cuda_idx"] = wrap_arg_val(least_used_gpu)
            args["cuda_idx_predictor"] = wrap_arg_val(least_used_gpu)

            p = start_process(convert_args(args))
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                print(f"Process {p.pid} finished with exit code {p.exitcode}. Running: {len(running_processes)-1}, queued: {len(process_args)}, failed: {len(failed_processes)}")
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--communication_model', 'bernoulli', '--communication_probability', '0.0', '--critic_hidden_size', '64', '--critic_lr', '0.0005', '--cuda', '--cuda_deterministi

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '50', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '50', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--communication_model', 'bernoulli', '--communication_probabil

...

QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
R_Critic: Use CNN: False, Use MLP: True
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-draft1/wandb/run-20260513_145517-q92oa1om/files
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Initialized QmasAlgorithm with 1 predictors. Threaded training: True
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
R_Critic: Use CNN: False, Use MLP: True
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-draft1/wandb/run-20260513_212509-zyxc9sde/files
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Buffer initialized with episode length 150
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-draft1/wandb/run-20260513_212509-zyxc9sde/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolli

# Analysis

In [4]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To load results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

/
 ├── MAGEC
 │   ├── obs_radius_10.0_comms_0.000
 │   │   ├── agent_count (10, 150) float32
 │   │   ├── avg_idleness (10, 150) float32
 │   │   ├── communication
 │   │   │   └── requests_made (10, 150) float32
 │   │   ├── deltaSteps (10, 150, 4, 1) float32
 │   │   ├── node_visits
 │   │   │   ├── node_0 (10, 150) float32
 │   │   │   ├── node_1 (10, 150) float32
 │   │   │   ├── node_10 (10, 150) float32
 │   │   │   ├── node_11 (10, 150) float32
 │   │   │   ├── node_12 (10, 150) float32
 │   │   │   ├── node_13 (10, 150) float32
 │   │   │   ├── node_14 (10, 150) float32
 │   │   │   ├── node_15 (10, 150) float32
 │   │   │   ├── node_16 (10, 150) float32
 │   │   │   ├── node_17 (10, 150) float32
 │   │   │   ├── node_18 (10, 150) float32
 │   │   │   ├── node_19 (10, 150) float32
 │   │   │   ├── node_2 (10, 150) float32
 │   │   │   ├── node_20 (10, 150) float32
 │   │   │   ├── node_21 (10, 150) float32
 │   │   │   ├── node_22 (10, 150) float32
 │   │   │   ├── node_23 (10,

In [5]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# SERIES_TO_PLOT = {"Learned Comms. + Prediction + UQ", "MAGEC"}
SERIES_TO_PLOT = None # Plot all

series_colors = {s: c for s, c in zip(output, px.colors.qualitative.Plotly)}


def get_requests(data):
    """Total communication requests per episode (mean over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return float(np.mean(r.reshape(r.shape[0], -1).sum(axis=1)))


def get_idleness_mean(data):
    """Time-averaged avg. idleness, mean over episodes."""
    return float(np.mean(data['avg_idleness']))


def fit_ols_trendline(xs, ys):
    """
    Fit an OLS linear trendline y = a*x + b.
    Returns (x_smooth, y_smooth, r2).
    """
    xs, ys = np.array(xs, dtype=float), np.array(ys, dtype=float)
    coeffs = np.polyfit(xs, ys, 1)
    x_smooth = np.linspace(xs.min(), xs.max(), 200)
    y_smooth = np.polyval(coeffs, x_smooth)
    y_pred = np.polyval(coeffs, xs)
    ss_res = np.sum((ys - y_pred) ** 2)
    ss_tot = np.sum((ys - ys.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    return x_smooth, y_smooth, r2


for obs_radius in obs_radii:
    fig = go.Figure()
    r2_annotations = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        color = series_colors.get(series, '#888888')
        all_exps = list(output[series].keys())

        # Swept experiments: encoded as obs_radius_X_comms_Y
        swept_exps = sorted(
            [e for e in all_exps if f"obs_radius_{obs_radius:.1f}_comms_" in e],
            key=lambda e: float(e.split("_comms_")[1])
        )
        # Fixed (learned comms) experiments: encoded as obs_radius_X only
        fixed_exps = [e for e in all_exps if e == f"obs_radius_{obs_radius:.1f}"]

        if swept_exps:
            comms_ps = [float(e.split("_comms_")[1]) for e in swept_exps]
            xs = [get_requests(output[series][e]) for e in swept_exps]
            ys = [get_idleness_mean(output[series][e]) for e in swept_exps]

            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                hovertemplate=[
                    f"{series}<br>comms_p={cp:.3f}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>"
                    for cp in comms_ps
                ],
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend, y_trend, r2 = fit_ols_trendline(xs, ys)
                fig.add_trace(go.Scatter(
                    x=x_trend,
                    y=y_trend,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r2_annotations.append(
                    dict(
                        text=f"<b>{series}</b> R²={r2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r2_annotations) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed_exps:
            xs_f = [get_requests(output[series][e]) for e in fixed_exps]
            ys_f = [get_idleness_mean(output[series][e]) for e in fixed_exps]
            fig.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='markers',
                name=series,
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept_exps) == 0,
                hovertemplate=f"{series}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
            ))

    fig.update_layout(
        title=f"Time-Averaged Avg. Idleness vs. Communication Requests (obs_radius={obs_radius:.1f})",
        xaxis_title="Total Communication Requests (per episode)",
        yaxis_title="Time-Averaged Avg. Idleness (s)",
        height=600,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r2_annotations,
    )
    fig.show()


In [6]:
fig = go.Figure()

for series in output:
    if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
        continue

    color = series_colors.get(series, '#888888')
    all_exps_s = list(output[series].keys())

    all_requests = []
    for exp in all_exps_s:
        r = np.array(output[series][exp]['communication']['requests_made'][...])
        # r shape: (episodes, timesteps, agents) -> sum over agents, keep timesteps
        r2d = r.reshape(r.shape[0], r.shape[1], -1).sum(axis=2)  # (episodes, timesteps)
        all_requests.append(r2d)

    if all_requests:
        all_requests = np.concatenate(all_requests, axis=0)  # (total_episodes, timesteps)
        mean_req = all_requests.mean(axis=0)
        std_req = all_requests.std(axis=0)
        timesteps = np.arange(len(mean_req))

        fig.add_trace(go.Scatter(
            x=timesteps,
            y=mean_req,
            mode='lines',
            name=series,
            line=dict(color=color),
            legendgroup=series,
            hovertemplate=f"{series}<br>t=%{{x}}<br>requests=%{{y:.2f}}<extra></extra>",
        ))
        # Std band
        fig.add_trace(go.Scatter(
            x=np.concatenate([timesteps, timesteps[::-1]]).tolist(),
            y=np.concatenate([mean_req + std_req, (mean_req - std_req)[::-1]]).tolist(),
            fill='toself',
            fillcolor=color,
            line=dict(color='rgba(255,255,255,0)'),
            opacity=0.15,
            legendgroup=series,
            showlegend=False,
            hoverinfo='skip',
        ))

fig.update_layout(
    title="Communication Requests vs. Time",
    xaxis_title="Timestep",
    yaxis_title="Communication Requests (per timestep)",
    height=600,
    hovermode='closest',
    autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()


In [7]:
for comms_p in comms_probabilities:
    fig2 = go.Figure()
    r2_annotations2 = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        color = series_colors.get(series, '#888888')
        all_exps_s = list(output[series].keys())

        # Swept experiments for this comms_p: obs_radius_X_comms_Y
        swept = sorted(
            [e for e in all_exps_s if f"_comms_{comms_p:.3f}" in e],
            key=lambda e: float(e.split("obs_radius_")[1].split("_comms_")[0])
        )
        # Fixed (learned comms) experiments: obs_radius_X only (no comms component)
        fixed = sorted(
            [e for e in all_exps_s if not "_comms_" in e],
            key=lambda e: float(e.split("obs_radius_")[1])
        )

        if swept:
            xs = [float(e.split("obs_radius_")[1].split("_comms_")[0]) for e in swept]
            ys = [get_requests(output[series][e]) for e in swept]

            fig2.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                hovertemplate=[
                    f"{series}<br>obs_radius={r:.1f}<br>requests=%{{y:.1f}}<extra></extra>"
                    for r in xs
                ],
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend2, y_trend2, r2_2 = fit_ols_trendline(xs, ys)
                fig2.add_trace(go.Scatter(
                    x=x_trend2,
                    y=y_trend2,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r2_annotations2.append(
                    dict(
                        text=f"<b>{series}</b> R²={r2_2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r2_annotations2) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed:
            xs_f = [float(e.split("obs_radius_")[1]) for e in fixed]
            ys_f = [get_requests(output[series][e]) for e in fixed]
            fig2.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept) == 0,
                hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>requests=%{{y:.1f}}<extra></extra>",
            ))

    fig2.update_layout(
        title=f"Communication Requests vs. Observation Radius (comms_p={comms_p:.3f})",
        xaxis_title="Observation Radius",
        yaxis_title="Total Communication Requests (per episode)",
        height=600,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r2_annotations2,
    )
    fig2.show()


In [8]:
for comms_p in comms_probabilities:
    fig2 = go.Figure()
    r2_annotations2 = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        color = series_colors.get(series, '#888888')
        all_exps_s = list(output[series].keys())

        # Swept experiments for this comms_p: obs_radius_X_comms_Y
        swept = sorted(
            [e for e in all_exps_s if f"_comms_{comms_p:.3f}" in e],
            key=lambda e: float(e.split("obs_radius_")[1].split("_comms_")[0])
        )
        # Fixed (learned comms) experiments: obs_radius_X only (no comms component)
        fixed = sorted(
            [e for e in all_exps_s if not "_comms_" in e],
            key=lambda e: float(e.split("obs_radius_")[1])
        )

        if swept:
            xs = [float(e.split("obs_radius_")[1].split("_comms_")[0]) for e in swept]
            ys = [get_requests(output[series][e]) for e in swept]

            fig2.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                hovertemplate=[
                    f"{series}<br>obs_radius={r:.1f}<br>requests=%{{y:.1f}}<extra></extra>"
                    for r in xs
                ],
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend2, y_trend2, r2_2 = fit_ols_trendline(xs, ys)
                fig2.add_trace(go.Scatter(
                    x=x_trend2,
                    y=y_trend2,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r2_annotations2.append(
                    dict(
                        text=f"<b>{series}</b> R²={r2_2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r2_annotations2) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed:
            xs_f = [float(e.split("obs_radius_")[1]) for e in fixed]
            ys_f = [get_requests(output[series][e]) for e in fixed]
            fig2.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept) == 0,
                hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>requests=%{{y:.1f}}<extra></extra>",
            ))

    fig2.update_layout(
        title=f"Communication Requests vs. Observation Radius (comms_p={comms_p:.3f})",
        xaxis_title="Observation Radius",
        yaxis_title="Total Communication Requests (per episode)",
        height=600,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r2_annotations2,
    )
    fig2.show()
